### 2026-02-23 ERA5 1deg precipitation

Make a 1deg monthly precipitation file from the 0.25 degree data, following the other coarsened variables.

In [1]:
import xarray as xr
import xesmf

In [8]:
INPUT_PATH = '../local_data/ERA5/mon/ERA5_total_precipitation_mon_full_sfc_1940-2024.nc'
TEMPLATE_PATH = '../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_tas_1978-2024.nc'
OUTPUT_PATH = '../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_pr_1978-2024.nc'

In [9]:
precip_ds = xr.open_dataset(INPUT_PATH)
template_ds = xr.open_dataset(TEMPLATE_PATH)

/tmp/ipykernel_19558/2938706359.py:2: FutureWarning: In a future version, xarray will not decode the variable 'forecast_period' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  template_ds = xr.open_dataset(TEMPLATE_PATH)


In [10]:
template_ds

<xarray.Dataset> Size: 146MB
Dimensions:               (time: 564, lat: 180, lon: 360, bnds: 2)
Coordinates:
  * time                  (time) datetime64[ns] 5kB 1978-01-17 ... 2024-12-17
  * lat                   (lat) float64 1kB -89.5 -88.5 -87.5 ... 87.5 88.5 89.5
  * lon                   (lon) float64 3kB 0.5 1.5 2.5 ... 357.5 358.5 359.5
    forecast_period       timedelta64[ns] 8B ...
    height                float64 8B ...
    originating_centre    |S50 50B ...
Dimensions without coordinates: bnds
Data variables:
    tas                   (time, lat, lon) float32 146MB ...
    latitude_longitude    int32 4B ...
    time_bnds             (time, bnds) datetime64[ns] 9kB ...
    lat_bnds              (lat, bnds) float64 3kB ...
    lon_bnds              (lon, bnds) float64 6kB ...
    forecast_period_bnds  (bnds) float64 16B ...
Attributes:
    Conventions:  CF-1.7
    software:     Created with ESMValTool v2.14.0.dev68+g5a317f52f.d20260122

In [19]:
precip_ds.TP

<xarray.DataArray 'TP' (time: 1020, lat: 721, lon: 1440)> Size: 4GB
array([[[4.425049e-04, 4.425049e-04, ..., 4.425049e-04, 4.425049e-04],
        [4.692078e-04, 4.692078e-04, ..., 4.692078e-04, 4.692078e-04],
        ...,
        [1.258850e-04, 1.258850e-04, ..., 1.258850e-04, 1.258850e-04],
        [1.525879e-04, 1.525879e-04, ..., 1.525879e-04, 1.525879e-04]],

       [[3.604889e-04, 3.604889e-04, ..., 3.604889e-04, 3.604889e-04],
        [3.604889e-04, 3.604889e-04, ..., 3.604889e-04, 3.604889e-04],
        ...,
        [7.247925e-05, 7.247925e-05, ..., 7.247925e-05, 7.247925e-05],
        [9.536743e-05, 9.536743e-05, ..., 9.536743e-05, 9.536743e-05]],

       ...,

       [[7.286072e-04, 7.286072e-04, ..., 7.286072e-04, 7.286072e-04],
        [6.656647e-04, 6.656647e-04, ..., 6.656647e-04, 6.656647e-04],
        ...,
        [1.993179e-04, 1.993179e-04, ..., 1.993179e-04, 1.993179e-04],
        [2.470016e-04, 2.470016e-04, ..., 2.470016e-04, 2.470016e-04]],

       [[5.970001e-04, 5.970001e-04, ..., 5.970001e-04, 5.970001e-04],
        [5.912781e-04, 5.912781e-04, ..., 5.912781e-04, 5.912781e-04],
        ...,
        [4.005432e-05, 4.005432e-05, ..., 4.005432e-05, 4.005432e-05],
        [6.484985e-05, 6.484985e-05, ..., 6.484985e-05, 6.484985e-05]]],
      shape=(1020, 721, 1440), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 8kB 1940-01-01 1940-02-01 ... 2024-12-01
  * lon      (lon) float64 12kB 0.0 0.25 0.5 0.75 ... 359.0 359.2 359.5 359.8
  * lat      (lat) float64 6kB 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
Attributes:
    long_name:  Total precipitation
    units:      m
    code:       228
    table:      128

In [15]:
regridder = xesmf.Regridder(
    precip_ds.TP,
    template_ds[['lat', 'lon']],
    method='conservative'
)

/home/brianhenn/miniconda3/envs/fme/lib/python3.11/site-packages/xesmf/backend.py:57: UserWarning: Latitude is outside of [-90, 90]
  warnings.warn('Latitude is outside of [-90, 90]')
/home/brianhenn/miniconda3/envs/fme/lib/python3.11/site-packages/xesmf/frontend.py:96: UserWarning: Variables {'lon_bnds'} not found in object but are referred to in the CF attributes.
  lon_bnds = ds.cf.get_bounds('longitude')


In [16]:
precip_regridded = regridder.regrid_dataarray(precip_ds.TP)

In [20]:
precip_regridded

<xarray.DataArray (time: 1020, lat: 180, lon: 360)> Size: 264MB
array([[[1.47343380e-04, 1.46874503e-04, 1.46940569e-04, ...,
         1.46835926e-04, 1.46527047e-04, 1.46644539e-04],
        [1.30008266e-04, 1.29684195e-04, 1.29144100e-04, ...,
         1.29700420e-04, 1.29892811e-04, 1.30008266e-04],
        [1.24603670e-04, 1.24529877e-04, 1.24402970e-04, ...,
         1.23780759e-04, 1.24166792e-04, 1.24442711e-04],
        ...,
        [6.52052171e-04, 6.54722389e-04, 6.57368801e-04, ...,
         6.52665272e-04, 6.51981682e-04, 6.51637092e-04],
        [6.07486581e-04, 6.09814771e-04, 6.12181437e-04, ...,
         6.01477688e-04, 6.03426015e-04, 6.05294888e-04],
        [5.38386172e-04, 5.36861480e-04, 5.35963511e-04, ...,
         5.36045118e-04, 5.35280851e-04, 5.36035339e-04]],

       [[6.28578273e-05, 6.25349785e-05, 6.23117594e-05, ...,
         6.25406756e-05, 6.23476226e-05, 6.24210516e-05],
        [4.41273660e-05, 4.41273660e-05, 4.41273660e-05, ...,
         4.42812816e-05, 4.42812816e-05, 4.42620367e-05],
        [5.51057128e-05, 5.47149975e-05, 5.43507304e-05, ...,
         5.62333007e-05, 5.58835418e-05, 5.56004161e-05],
...
         8.10597092e-04, 8.04495008e-04, 7.98942172e-04],
        [7.11434870e-04, 7.05659157e-04, 7.00119941e-04, ...,
         7.27355713e-04, 7.22086988e-04, 7.16957380e-04],
        [7.01301498e-04, 6.97409152e-04, 6.95646391e-04, ...,
         7.02562567e-04, 6.99700380e-04, 6.99356489e-04]],

       [[3.14302997e-05, 3.09516836e-05, 3.09417665e-05, ...,
         3.20975378e-05, 3.13403834e-05, 3.13009077e-05],
        [1.64074645e-05, 1.63018813e-05, 1.61864373e-05, ...,
         1.87556179e-05, 1.82092299e-05, 1.76169106e-05],
        [2.25018775e-05, 2.23269126e-05, 2.21994651e-05, ...,
         2.53969520e-05, 2.45142219e-05, 2.34205745e-05],
        ...,
        [6.42635277e-04, 6.44632557e-04, 6.45653054e-04, ...,
         6.33889809e-04, 6.37124060e-04, 6.40295097e-04],
        [6.38561207e-04, 6.37333083e-04, 6.36071374e-04, ...,
         6.37714926e-04, 6.38307130e-04, 6.38598693e-04],
        [6.46388158e-04, 6.43477251e-04, 6.42235565e-04, ...,
         6.45201129e-04, 6.43659208e-04, 6.44114916e-04]]],
      shape=(1020, 180, 360), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 8kB 1940-01-01 1940-02-01 ... 2024-12-01
  * lat      (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon      (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
Attributes:
    regrid_method:  conservative